In [1]:
!wget https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip

--2026-07-18 19:49:39--  https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt [following]
--2026-07-18 19:49:39--  https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc0eb47dc36844e666151dfa49ee.dl.dropboxusercontent.com/cd/0/inline/DEgfT8SccakFzpqKe7q0Gc9TXDducdtwLbG-oQPifb3K71B3JzNX8AoQ8BqyMX5lYYmqLVLX_V0x8dnLMOyYp3UZLctTB7Jp4K6c7MaWPy3nWtb29JMDspLLj-m6x_4Juef3xgMF9sCNH82TrtdRS69S/file# [following]
--2026-07-18 19:49:40--  https://uc0eb47dc36844e666151dfa49ee.

In [2]:
!unzip -q "/content/movie-reviews-dataset.zip"

In [3]:
from tensorflow.keras.preprocessing import text_dataset_from_directory
from tensorflow.strings import regex_replace
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout

In [4]:
def prepareData(dir):
  data = text_dataset_from_directory(dir)
  return data.map(
    lambda text, label: (regex_replace(text, '<br />', ' '), label),
  )

In [5]:
train_data = prepareData('movie-reviews-dataset/train')
test_data = prepareData('movie-reviews-dataset/test')

for text_batch, label_batch in train_data.take(1):
  print(text_batch.numpy()[0])
  print(label_batch.numpy()[0])

Found 25000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.
b"Being a fan of cheesy horror movies, I saw this in my video shop and thought I would give it a try. Now that I've seen it I wish it upon no living soul on the planet. I get my movie rentals for free, and I feel that I didn't get my moneys worth. I've seen some bad cheesy horror movies in my time, hell I'm a fan of them, but this was just an insult."
0


In [6]:
model = Sequential()

In [7]:
model.add(Input(shape=(), dtype="string"))

In [8]:
max_tokens = 1000
max_len = 100
vectorize_layer = TextVectorization(
  max_tokens=max_tokens,
  output_mode="int",
  output_sequence_length=max_len,
)

In [9]:
train_texts = train_data.map(lambda text, label: text)
vectorize_layer.adapt(train_texts)

model.add(vectorize_layer)

In [10]:
model.add(Embedding(max_tokens + 1, 128))

model.add(LSTM(64))
model.add(Dense(64, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

In [11]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [12]:
model.fit(train_data, epochs=10)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.6029 - loss: 0.6623
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.6943 - loss: 0.5889
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.7756 - loss: 0.4774
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8036 - loss: 0.4284
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8144 - loss: 0.4038
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8281 - loss: 0.3779
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8389 - loss: 0.3609
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8481 - loss: 0.3406
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.8554 - loss: 0.3238
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8666 - loss: 0.3051


In [13]:
model.evaluate(test_data)

782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7858 - loss: 0.4960


[0.4959854781627655, 0.7858399748802185]

In [30]:
text = "I love that movie !"

In [31]:
import tensorflow as tf
model.predict(tf.constant([text]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


array([[0.8771543]], dtype=float32)